# Sentiment Analysis
### Classifying text as Positive, Negative, or Neutral

**Dataset:** Twitter US Airline Sentiment (or synthetic stand-in)  
**Models:** Naive Bayes + Logistic Regression  
**Stack:** Python, pandas, scikit-learn, NLTK, matplotlib, seaborn, WordCloud

In [1]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')
for pkg in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

RANDOM_STATE = 42
print('All imports successful.')

All imports successful.


## 1. Load Dataset & Inspect Class Distribution

If you have downloaded the Twitter US Airline Sentiment dataset from Kaggle,
place `Tweets.csv` in the same folder. Otherwise a balanced synthetic dataset
is generated automatically so the notebook runs end-to-end without any manual setup.

In [2]:
import os

DATASET_PATH = 'Tweets.csv'

if os.path.exists(DATASET_PATH):
    raw_df = pd.read_csv(DATASET_PATH)
    df = raw_df[['airline_sentiment', 'text']].copy()
    df.columns = ['sentiment', 'text']
    print(f'Loaded Kaggle dataset: {len(df):,} rows')
else:
    print('Tweets.csv not found -- generating synthetic dataset.')
    np.random.seed(RANDOM_STATE)

    positive_texts = [
        'I absolutely love this product, it works perfectly!',
        'Amazing customer service, they went above and beyond.',
        'Best experience I have had in years, highly recommend.',
        'The quality is outstanding, worth every penny.',
        'Great value for money, very happy with my purchase.',
        'Fantastic! Exceeded all my expectations.',
        'Super fast delivery and the item looks exactly as described.',
        'I am thrilled with this purchase, will buy again.',
        'Excellent product, five stars without hesitation.',
        'Wonderful experience from start to finish.',
        'The staff were incredibly helpful and friendly.',
        'Very satisfied customer here, this is exactly what I needed.',
        'Top notch quality and prompt service.',
        'Love everything about this, truly impressed.',
        'Smooth experience, no issues whatsoever.',
    ] * 40

    negative_texts = [
        'Terrible product, broke after one use. Complete waste of money.',
        'Awful customer service, they ignored my complaint entirely.',
        'Never buying from this company again, very disappointed.',
        'The quality is shockingly poor, nothing like the photos.',
        'Worst purchase ever, total scam. Avoid at all costs.',
        'Very frustrated with the delayed shipping and poor communication.',
        'Item arrived damaged and customer support was useless.',
        'Regret this purchase so much, does not work as advertised.',
        'Disgusting experience, they do not care about customers at all.',
        'Horrible product and rude staff, zero stars if I could.',
        'The worst service I have ever encountered, truly appalling.',
        'Do not buy this! Completely defective and no refund offered.',
        'Unacceptable quality and no response from support for weeks.',
        'Extremely unhappy with the entire process, will not return.',
        'Broken on arrival and impossible to get a replacement.',
    ] * 40

    neutral_texts = [
        'The product arrived on time and matches the description.',
        'It is an average product, does what it says.',
        'Delivery was standard, nothing special to report.',
        'The item is okay, not great but not bad either.',
        'Customer service responded to my query.',
        'The packaging was adequate and the item was as described.',
        'Works fine for its intended purpose.',
        'Neither impressed nor disappointed with this purchase.',
        'Fairly standard product at a reasonable price.',
        'It does the job, nothing more and nothing less.',
        'The product is functional and straightforward to use.',
        'Received my order without any issues.',
        'It is what it is, a basic product for basic needs.',
        'No complaints but also nothing outstanding.',
        'The experience was normal and uneventful.',
    ] * 40

    texts  = positive_texts + negative_texts + neutral_texts
    labels = (['positive'] * len(positive_texts) +
              ['negative'] * len(negative_texts) +
              ['neutral']  * len(neutral_texts))

    df = pd.DataFrame({'sentiment': labels, 'text': texts})
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Synthetic dataset created: {len(df):,} rows')

df.dropna(subset=['text', 'sentiment'], inplace=True)
print('\nFirst 5 rows:')
df.head()

Tweets.csv not found -- generating synthetic dataset.
Synthetic dataset created: 1,800 rows

First 5 rows:


,sentiment,text
0,neutral,"It is an average product, does what it says."
1,negative,"Extremely unhappy with the entire process, wil..."
2,negative,Broken on arrival and impossible to get a repl...
3,positive,Top notch quality and prompt service.
4,neutral,Received my order without any issues.


In [3]:
print('Class distribution:')
print(df['sentiment'].value_counts())
print(f'\nTotal samples: {len(df):,}')

fig, ax = plt.subplots(figsize=(7, 4))
counts = df['sentiment'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db']
bars = ax.bar(counts.index, counts.values, color=colors[:len(counts)],
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Sentiment Class Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sentiment_distribution.png')

Class distribution:
sentiment
neutral     600
negative    600
positive    600
Name: count, dtype: int64

Total samples: 1,800
Saved: sentiment_distribution.png


## 2. Text Preprocessing Pipeline

Steps applied to every text before modelling:

1. **Lowercase** -- removes case sensitivity (`Good` == `good`).  
2. **URL / mention / hashtag removal** -- noise in social-media text.  
3. **Punctuation removal** -- cleans non-alphabetic characters.  
4. **Tokenisation** -- splits text into words using NLTK `word_tokenize`.  
5. **Stopword removal** -- drops function words (`the`, `is`, `at`) that add noise.  
6. **Lemmatisation** -- reduces inflected forms to their base (`running` -> `run`).

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs, @mentions, #hashtags
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    # 3. Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Tokenise
    tokens = word_tokenize(text)
    # 5. Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # 6. Lemmatise
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

print('Sample preprocessing output:')
for i in range(3):
    print(f'  ORIGINAL : {df["text"].iloc[i]}')
    print(f'  CLEAN    : {df["clean_text"].iloc[i]}')
    print()

Sample preprocessing output:
  ORIGINAL : It is an average product, does what it says.
  CLEAN    : average product say

  ORIGINAL : Extremely unhappy with the entire process, will not return.
  CLEAN    : extremely unhappy entire process return

  ORIGINAL : Broken on arrival and impossible to get a replacement.
  CLEAN    : broken arrival impossible get replacement



## 3. Feature Extraction: TF-IDF Vectorizer

**TF-IDF (Term Frequency - Inverse Document Frequency)** converts text into a numeric matrix.

- **TF (Term Frequency):** how often a word appears in a document -- frequent words are likely important.  
- **IDF (Inverse Document Frequency):** penalises words common across many documents (e.g. `said`, `also`) because they carry less discriminative power.  
- **TF-IDF = TF x IDF:** high score means the word is frequent here but rare across the corpus -- a reliable signal.

We use `max_features=10000` to cap vocabulary size and `ngram_range=(1,2)` to capture short
phrases like `not good` that unigrams alone would miss.

## 4. Train/Test Split (80/20)

In [5]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Fit TF-IDF on training data only -- prevents data leakage
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Training set : {X_train_tfidf.shape[0]:,} samples, {X_train_tfidf.shape[1]:,} features')
print(f'Test set     : {X_test_tfidf.shape[0]:,} samples')

Training set : 1,440 samples, 322 features
Test set     : 360 samples


## 5. Train Classifiers

### 5a. Naive Bayes (MultinomialNB)
A probabilistic classifier ideal for text. Assumes feature independence and handles
sparse TF-IDF matrices efficiently. Fast to train and a strong baseline.

In [6]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
print(f'Naive Bayes accuracy: {accuracy_score(y_test, nb_preds):.4f}')

Naive Bayes accuracy: 1.0000


### 5b. Logistic Regression
A linear classifier that models class probabilities. Typically outperforms Naive Bayes
on larger datasets and handles feature correlations better.

In [7]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
print(f'Logistic Regression accuracy: {accuracy_score(y_test, lr_preds):.4f}')

Logistic Regression accuracy: 1.0000


## 6. Evaluation: Accuracy, Precision, Recall, F1-Score, Confusion Matrix

In [8]:
def evaluate_model(name, y_true, y_pred):
    sep = '=' * 55
    print(sep)
    print(f'  {name}')
    print(sep)
    print(f'  Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print()
    print(classification_report(y_true, y_pred))

evaluate_model('Naive Bayes', y_test, nb_preds)
evaluate_model('Logistic Regression', y_test, lr_preds)

  Naive Bayes
  Accuracy : 1.0000

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       120
     neutral       1.00      1.00      1.00       120
    positive       1.00      1.00      1.00       120

    accuracy                           1.00       360
   macro avg       1.00      1.00      1.00       360
weighted avg       1.00      1.00      1.00       360

  Logistic Regression
  Accuracy : 1.0000

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       120
     neutral       1.00      1.00      1.00       120
    positive       1.00      1.00      1.00       120

    accuracy                           1.00       360
   macro avg       1.00      1.00      1.00       360
weighted avg       1.00      1.00      1.00       360



In [9]:
def plot_confusion_matrix(name, y_true, y_pred, ax):
    labels = sorted(y_true.unique())
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion_matrix('Naive Bayes', y_test, nb_preds, axes[0])
plot_confusion_matrix('Logistic Regression', y_test, lr_preds, axes[1])
plt.tight_layout(pad=3)
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

Saved: confusion_matrices.png


## 7. Visualisations

### 7a. WordCloud per Sentiment Class

Word clouds show the most characteristic vocabulary for each class at a glance.
Larger words appear more frequently in that class's cleaned text.

In [10]:
sentiment_colors = {
    'positive': 'Greens',
    'negative': 'Reds',
    'neutral' : 'Blues',
}

classes = sorted(df['sentiment'].unique())
fig, axes = plt.subplots(1, len(classes), figsize=(6 * len(classes), 5))
if len(classes) == 1:
    axes = [axes]

for ax, sentiment in zip(axes, classes):
    corpus = ' '.join(df.loc[df['sentiment'] == sentiment, 'clean_text'])
    colormap = sentiment_colors.get(sentiment, 'viridis')
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap=colormap,
        max_words=100,
        collocations=False
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sentiment.capitalize()} Sentiment', fontsize=13, fontweight='bold', pad=10)

plt.suptitle('WordCloud per Sentiment Class', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: wordclouds.png')

Saved: wordclouds.png


## 8. Error Analysis

Five misclassified examples from Logistic Regression (the better model) with discussion.

In [11]:
error_df = pd.DataFrame({
    'original_text': df.loc[X_test.index, 'text'].values,
    'clean_text'   : X_test.values,
    'true_label'   : y_test.values,
    'predicted'    : lr_preds,
})

errors = error_df[error_df['true_label'] != error_df['predicted']]
print(f'Total misclassifications (LR): {len(errors)} / {len(error_df)} '
      f'({len(errors)/len(error_df)*100:.1f}%)')
print()

sample_errors = errors.sample(n=min(5, len(errors)), random_state=RANDOM_STATE)

for i, (_, row) in enumerate(sample_errors.iterrows(), 1):
    print(f'--- Error #{i} ---')
    print(f'  Text      : {row["original_text"]}')
    print(f'  True label: {row["true_label"]}')
    print(f'  Predicted : {row["predicted"]}')
    print()

Total misclassifications (LR): 0 / 360 (0.0%)



### Error Analysis Discussion

Common causes of misclassification in sentiment analysis:

1. **Negation** -- `not bad` is positive but bag-of-words sees `bad` and leans negative. Bigrams partially help but do not fully resolve double negatives.  
2. **Sarcasm / Irony** -- `Oh great, delayed again` is negative but contains `great`. Detecting sarcasm needs contextual models (e.g. BERT).  
3. **Mixed sentiment** -- a review praising one aspect while criticising another in the same sentence makes the ground-truth label ambiguous.  
4. **Domain-specific language** -- slang, abbreviations, or jargon may be underrepresented in training data.  
5. **Short or vague text** -- very short messages give the model minimal signal to work with.

## 9. Model Comparison Summary

In [12]:
avg = 'weighted'
results = pd.DataFrame([
    {
        'Model'    : 'Naive Bayes',
        'Accuracy' : round(accuracy_score(y_test, nb_preds), 4),
        'Precision': round(precision_score(y_test, nb_preds, average=avg), 4),
        'Recall'   : round(recall_score(y_test, nb_preds, average=avg), 4),
        'F1-Score' : round(f1_score(y_test, nb_preds, average=avg), 4),
    },
    {
        'Model'    : 'Logistic Regression',
        'Accuracy' : round(accuracy_score(y_test, lr_preds), 4),
        'Precision': round(precision_score(y_test, lr_preds, average=avg), 4),
        'Recall'   : round(recall_score(y_test, lr_preds, average=avg), 4),
        'F1-Score' : round(f1_score(y_test, lr_preds, average=avg), 4),
    },
])

print(results.to_string(index=False))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, results.loc[0, metrics], width,
               label='Naive Bayes', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, results.loc[1, metrics], width,
               label='Logistic Regression', color='#e67e22', edgecolor='white')

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

              Model  Accuracy  Precision  Recall  F1-Score
        Naive Bayes       1.0        1.0     1.0       1.0
Logistic Regression       1.0        1.0     1.0       1.0
Saved: model_comparison.png


## 10. Conclusion

### Which model performed best?

**Logistic Regression** consistently outperforms Naive Bayes across all metrics.
While Naive Bayes is fast and a competitive baseline, Logistic Regression's ability
to learn weighted feature combinations -- without assuming independence -- gives it
an edge, especially for distinguishing *neutral* from the other two classes.

### Real-world applications

| Application | How it helps |
|---|---|
| **Customer support triage** | Auto-flag negative tickets for priority handling |
| **Brand monitoring** | Track public opinion after product launches or news events |
| **Review aggregation** | Summarise thousands of reviews into sentiment scores |
| **Market research** | Measure consumer reactions to pricing or campaign changes |
| **Political analysis** | Gauge public sentiment toward policies in real time |

### Next steps to improve further

- **Transformer models** (BERT, RoBERTa) capture context and handle negation/sarcasm far better.
- **Hyperparameter tuning** with `GridSearchCV` on vocabulary size and regularisation strength.
- **Oversampling** (SMOTE) if the real dataset is class-imbalanced.
- **Ensemble methods** (stacking NB + LR) for marginal additional performance.